# Pitch Mix Tool
Grade any pitcher's arsenal against league average and flag where usage and results don't match.

## How it works
1. **Load** any pitcher's 2026 regular-season pitches from Statcast.
2. **Compare** against a league baseline built from every 2026 pitch.
3. **Grade** each pitch by matchup (pitch type, pitcher hand, batter side) on whiff% and run value.
4. **Flag** pitches where usage and results don't line up.

In [ ]:
!pip install pybaseball

In [ ]:
import unicodedata
from pybaseball import chadwick_register, statcast_pitcher

def strip_accents(text):
    text = unicodedata.normalize('NFKD', str(text))
    return ''.join(c for c in text if not unicodedata.combining(c)).lower().strip()

REGISTER = chadwick_register()
REGISTER['last_clean'] = REGISTER['name_last'].map(strip_accents)
REGISTER['first_clean'] = REGISTER['name_first'].map(strip_accents)

def find_player_id(first, last):
    matches = REGISTER[(REGISTER['last_clean'] == strip_accents(last)) &
                       (REGISTER['first_clean'] == strip_accents(first)) &
                       (REGISTER['mlb_played_last'].notna())]
    if matches.empty:
        return None
    matches = matches.sort_values('mlb_played_last', ascending=False)
    return int(matches['key_mlbam'].iloc[0])

def get_pitcher_data(first, last, season):
    player_id = find_player_id(first, last)
    if player_id is None:
        print("No player found. Check the spelling.")
        return None

    df = statcast_pitcher(f'{season}-03-01', f'{season}-11-30', player_id=player_id)
    df = df[df['game_type'] == 'R'].copy()

    print(f"{first.title()} {last.title()} ({season}): {len(df)} regular-season pitches")
    return df

Gathering player lookup table. This may take a moment.


In [ ]:
senga = get_pitcher_data('kodai', 'senga', 2025)
skubal = get_pitcher_data('tarik', 'skubal', 2026)

Gathering Player Data
Kodai Senga (2025): 1908 regular-season pitches
Gathering Player Data
Tarik Skubal (2026): 2357 regular-season pitches


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## The league baseline
Every 2026 regular-season pitch (716,792), grouped by pitch type, pitcher hand and batter side.
- **Whiff%** = misses ÷ swings
- **Run value** uses Statcast's `delta_run_exp`, flipped so positive means the pitch helped the pitcher, reported per 100 pitches
- **Whiff+** = pitcher's whiff% ÷ league whiff% × 100 (100 = league average)

## Location
- **Zone%:** share of pitches in the strike zone (Statcast zones 1–9)
- **Chase%:** swing rate on pitches outside the zone
- **In-zone whiff%:** misses on swings at strikes

Each is compared to league average for the same pitch type, pitcher hand and batter side, in percentage points.

In [ ]:
import os
import pandas as pd
pd.set_option('display.max_colwidth', None)
from pybaseball import statcast

KEEP = ['game_pk', 'game_date', 'game_type', 'pitcher', 'player_name', 'p_throws', 'stand',
        'pitch_type', 'balls', 'strikes', 'description', 'events', 'delta_run_exp',
        'release_speed', 'pfx_x', 'pfx_z', 'at_bat_number', 'pitch_number',
        'launch_speed', 'launch_angle', 'estimated_woba_using_speedangle', 'bb_type',
        'zone', 'plate_x', 'plate_z', 'sz_top', 'sz_bot']

SAVE_DIR = '/content/drive/MyDrive/pitch_mix_tool'
os.makedirs(SAVE_DIR, exist_ok=True)

months = [('2026-03-15', '2026-04-30'), ('2026-05-01', '2026-05-31'),
          ('2026-06-01', '2026-06-30'), ('2026-07-01', '2026-07-31'),
          ('2026-08-01', '2026-08-31'), ('2026-09-01', '2026-09-30')]

for start, end in months:
    path = f'{SAVE_DIR}/league_v3_{start[:7]}.parquet'
    if os.path.exists(path):
        print(f'{start[:7]} already saved, skipping')
        continue
    chunk = statcast(start, end)
    chunk = chunk[chunk['game_type'] == 'R']
    chunk = chunk[[c for c in KEEP if c in chunk.columns]]
    chunk.to_parquet(path)
    print(f'{start[:7]}: {len(chunk):,} pitches saved')

2026-03 already saved, skipping
2026-05 already saved, skipping
2026-06 already saved, skipping
2026-07 already saved, skipping
2026-08 already saved, skipping
2026-09 already saved, skipping


In [ ]:
import glob
league = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(f'{SAVE_DIR}/league_v3_*.parquet'))],
                   ignore_index=True)
print(f'{len(league):,} league pitches')
league['pitch_type'].value_counts().head(12)

716,792 league pitches


,count
pitch_type,
FF,219170
SI,118801
SL,93540
CH,79841
ST,59340
FC,57248
CU,45409
FS,23080
KC,11689


In [ ]:
league.groupby('description')['delta_run_exp'].mean().loc[['ball', 'called_strike', 'swinging_strike']]

,delta_run_exp
description,
ball,0.061129
called_strike,-0.066291
swinging_strike,-0.112411


In [ ]:
SWINGS = ['swinging_strike', 'swinging_strike_blocked', 'foul', 'foul_tip',
          'hit_into_play', 'foul_bunt', 'missed_bunt']
WHIFFS = ['swinging_strike', 'swinging_strike_blocked']

def add_flags(df):
    df = df.copy()
    df['is_swing'] = df['description'].isin(SWINGS)
    df['is_whiff'] = df['description'].isin(WHIFFS)
    df['rv_pitcher'] = -df['delta_run_exp']
    df['is_bip'] = df['description'] == 'hit_into_play'
    df['ev_bip'] = df['launch_speed'].where(df['is_bip'])
    df['xwoba_bip'] = df['estimated_woba_using_speedangle'].where(df['is_bip'])
    df['is_hard_hit'] = df['is_bip'] & (df['launch_speed'] >= 95).fillna(False)
    df['ivb'] = df['pfx_z'] * 12
    df['hb_arm'] = df['pfx_x'] * 12 * df['p_throws'].map({'R': -1, 'L': 1})
    df['count_state'] = 'Even'
    df.loc[df['strikes'] > df['balls'], 'count_state'] = 'Ahead'
    df.loc[df['balls'] > df['strikes'], 'count_state'] = 'Behind'
    if 'zone' in df.columns:
        df['in_zone'] = df['zone'].between(1, 9).fillna(False)
        df['out_zone'] = (df['zone'] >= 11).fillna(False)
        df['is_chase'] = df['is_swing'] & df['out_zone']
        df['z_swing'] = df['is_swing'] & df['in_zone']
        df['z_whiff'] = df['is_whiff'] & df['in_zone']
    return df

league = add_flags(league)

In [ ]:
def summarize(df, by):
    g = df.groupby(by)
    out = pd.DataFrame({
        'pitches': g.size(),
        'swings': g['is_swing'].sum(),
        'whiffs': g['is_whiff'].sum(),
        'rv_per_100': g['rv_pitcher'].mean() * 100,
        'bip': g['is_bip'].sum(),
        'hard_hits': g['is_hard_hit'].sum(),
        'avg_ev': g['ev_bip'].mean(),
        'velo': g['release_speed'].mean(),
        'ivb': g['ivb'].mean(),
        'hb_arm': g['hb_arm'].mean(),
        'in_zone': g['in_zone'].sum(),
        'out_zone': g['out_zone'].sum(),
        'chases': g['is_chase'].sum(),
        'z_swings': g['z_swing'].sum(),
        'z_whiffs': g['z_whiff'].sum(),
    })
    out['whiff_pct'] = out['whiffs'] / out['swings'] * 100
    out['hard_hit_pct'] = out['hard_hits'] / out['bip'] * 100
    out['zone_pct'] = out['in_zone'] / (out['in_zone'] + out['out_zone']) * 100
    out['chase_pct'] = out['chases'] / out['out_zone'] * 100
    out['z_whiff_pct'] = out['z_whiffs'] / out['z_swings'] * 100
    out = out.round(2)
    out['xwobacon'] = g['xwoba_bip'].mean().round(3)
    return out

baseline = summarize(league, ['pitch_type', 'p_throws', 'stand'])
baseline.to_parquet(f'{SAVE_DIR}/baseline_2026.parquet')
baseline.loc['ST']

pitches  swings  whiffs  rv_per_100   bip  hard_hits  avg_ev  \
p_throws stand                                                                 
L        L         8706    4100    1288        0.34  1395        354   83.33   
         R         8001    3389     874       -0.56  1090        343    86.0   
R        L        15962    6737    1843       -0.19  2100        779   87.75   
         R        26671   12396    3836        0.15  4455       1261   85.15   

                 velo   ivb  hb_arm  ...  out_zone  chases  z_swings  \
p_throws stand                       ...                               
L        L      81.04  0.69  -14.06  ...      5162    1681      2419   
         R      80.74  0.72  -13.48  ...      4371    1320      2069   
R        L      83.52  0.82  -13.51  ...      8450    2442      4295   
         R      83.59   1.2  -13.46  ...     16020    5383      7013   

                z_whiffs  whiff_pct  hard_hit_pct  zone_pct  chase_pct  \
p_throws stand                                                           
L        L           418      31.41         25.38     40.71      32.56   
         R           270      25.79         31.47     45.37       30.2   
R        L           633      27.36          37.1     47.06       28.9   
         R          1058      30.95         28.31     39.93       33.6   

                z_whiff_pct  xwobacon  
p_throws stand                         
L        L            17.28     0.322  
         R            13.05     0.363  
R        L            14.74     0.379  
         R            15.09     0.325  

[4 rows x 21 columns]

In [ ]:
def grade_pitcher(df):
    df = add_flags(df)
    mine = summarize(df, ['pitch_type', 'p_throws', 'stand'])
    report = mine.join(baseline, rsuffix='_lg')

    report['usage_pct'] = (mine['pitches'] / mine.groupby(level='stand')['pitches'].transform('sum') * 100).round(1)
    report['whiff_plus'] = (report['whiff_pct'] / report['whiff_pct_lg'] * 100).round(0)
    report['rv_vs_lg'] = (report['rv_per_100'] - report['rv_per_100_lg']).round(2)
    report['xwobacon_vs_lg'] = (report['xwobacon'] - report['xwobacon_lg']).round(3)
    report['zone_vs_lg'] = (report['zone_pct'] - report['zone_pct_lg']).round(1)
    report['chase_vs_lg'] = (report['chase_pct'] - report['chase_pct_lg']).round(1)
    report['z_whiff_vs_lg'] = (report['z_whiff_pct'] - report['z_whiff_pct_lg']).round(1)
    report['small_sample'] = report['swings'] < 30

    report = report[report['pitches'] >= 10]

    cols = ['pitches', 'usage_pct', 'velo', 'ivb', 'hb_arm',
            'swings', 'whiff_pct', 'whiff_pct_lg', 'whiff_plus',
            'rv_per_100', 'rv_per_100_lg', 'rv_vs_lg',
            'bip', 'hard_hit_pct', 'hard_hit_pct_lg', 'avg_ev', 'xwobacon', 'xwobacon_lg', 'xwobacon_vs_lg',
            'zone_pct', 'zone_pct_lg', 'zone_vs_lg', 'chase_pct', 'chase_pct_lg', 'chase_vs_lg',
            'z_whiff_pct', 'z_whiff_pct_lg', 'z_whiff_vs_lg',
            'small_sample']
    return report[cols].reset_index(level='p_throws', drop=True)

In [ ]:
count_baseline = summarize(league, ['pitch_type', 'p_throws', 'count_state'])

def count_report(df, wide=True):
    df = add_flags(df)
    mine = summarize(df, ['pitch_type', 'p_throws', 'count_state'])
    rep = mine.join(count_baseline, rsuffix='_lg')

    rep['usage_pct'] = (mine['pitches'] / mine.groupby(level='count_state')['pitches'].transform('sum') * 100).round(1)
    rep['whiff_plus'] = (rep['whiff_pct'] / rep['whiff_pct_lg'] * 100).round(0)
    rep['rv_vs_lg'] = (rep['rv_per_100'] - rep['rv_per_100_lg']).round(2)
    rep['small_sample'] = rep['swings'] < 30
    rep = rep[rep['pitches'] >= 10]

    rep = rep[['pitches', 'usage_pct', 'swings', 'whiff_plus', 'rv_vs_lg', 'small_sample']].reset_index(level='p_throws', drop=True)
    if wide:
        return rep.unstack('count_state')
    return rep.reset_index()

count_report(skubal)

pitches               usage_pct              swings                \
count_state   Ahead Behind   Even     Ahead Behind  Even  Ahead Behind   Even   
pitch_type                                                                      
CH            264.0  103.0  236.0      31.5   22.9  22.1  141.0   67.0  152.0   
CU             58.0    NaN   56.0       6.9    NaN   5.2   30.0    NaN   17.0   
FF            326.0  171.0  399.0      38.9   38.0  37.4  193.0  104.0  217.0   
SI            115.0   92.0  205.0      13.7   20.4  19.2   62.0   45.0   82.0   
SL             75.0   80.0  172.0       8.9   17.8  16.1   39.0   36.0   71.0   

            whiff_plus               rv_vs_lg              small_sample  \
count_state      Ahead Behind   Even    Ahead Behind  Even        Ahead   
pitch_type                                                                
CH               166.0  119.0  152.0     0.11   3.09   1.3        False   
CU               126.0    NaN  140.0    -3.15   <NA>  2.37        False   
FF               120.0  112.0  120.0     0.68   1.61  1.49        False   
SI               184.0   64.0  127.0     1.32   0.62  0.02        False   
SL                71.0   65.0   93.0    -1.97   3.04  2.05        False   

                           
count_state Behind   Even  
pitch_type                 
CH           False  False  
CU             NaN   True  
FF           False  False  
SI           False  False  
SL           False  False

In [ ]:
skubal_report = grade_pitcher(skubal)
skubal_report.sort_values(['stand', 'usage_pct'], ascending=[True, False])

,,pitches,usage_pct,velo,ivb,hb_arm,swings,whiff_pct,whiff_pct_lg,whiff_plus,rv_per_100,...,zone_pct,zone_pct_lg,zone_vs_lg,chase_pct,chase_pct_lg,chase_vs_lg,z_whiff_pct,z_whiff_pct_lg,z_whiff_vs_lg,small_sample
pitch_type,stand,,,,,,,,,,,,,,,,,,,,,
SI,L,217,41.7,96.73,12.45,13.15,105,13.33,11.68,114.0,1.57,...,67.28,57.23,10.1,29.58,29.67,-0.1,9.52,7.83,1.7,False
FF,L,125,24.0,97.46,17.15,2.65,82,36.59,20.06,182.0,-1.19,...,60.00,54.86,5.1,40.00,26.02,14.0,32.26,15.72,16.5,False
CH,L,101,19.4,88.43,5.09,13.16,66,43.94,31.99,137.0,2.11,...,44.55,39.62,4.9,44.64,37.63,7.0,34.15,21.15,13.0,False
SL,L,41,7.9,89.66,3.55,-3.25,20,20.00,32.92,61.0,-5.77,...,56.10,42.72,13.4,22.22,32.33,-10.1,18.75,18.21,0.5,True
CU,L,36,6.9,80.81,-4.70,-8.49,19,52.63,29.46,179.0,3.76,...,47.22,38.01,9.2,52.63,29.39,23.2,22.22,13.61,8.6,True
FF,R,771,42.0,96.68,17.08,3.30,432,19.91,18.05,110.0,1.73,...,54.09,52.11,2.0,35.03,28.58,6.5,15.91,13.58,2.3,False
CH,R,502,27.3,87.50,4.75,13.78,294,46.94,30.66,153.0,0.80,...,31.27,35.44,-4.2,46.67,36.14,10.5,33.83,21.04,12.8,False
SL,R,286,15.6,89.74,4.12,-3.20,126,26.19,30.43,86.0,2.55,...,55.24,45.49,9.8,29.69,35.19,-5.5,15.91,16.0,-0.1,False
SI,R,195,10.6,96.69,12.24,13.25,84,19.05,11.59,164.0,0.04,...,49.74,55.7,-6.0,21.43,20.98,0.4,11.11,8.37,2.7,False


## Flag rules
A pitch can get more than one flag. Piles with under 30 swings are marked **Small sample** and skipped.

1. **Small sample: under 30 swings.** Below 30 swings, one or two whiffs move the rate by several points. At 20 swings, every whiff is worth 5 percentage points.
2. **Underused weapon: whiff+ ≥ 120, run value ≥ +1.0, usage under 25%.** The pitch beats league average on both measures but is a secondary pitch, so there's room to throw it more.
3. **Misses bats, still gets hurt: whiff+ ≥ 120, run value ≤ −1.0.** Whiffs alone can mislead. This catches pitches that miss bats but lose value on contact, walks or counts.
4. **Below league average: run value ≤ −1.5.** A stricter cutoff than the positive flags, because calling a pitch a problem is a stronger claim.
5. **Heavy use, below average: usage ≥ 25%, run value below 0.** A primary pitch that's costing runs, even slightly.
6. **Hit hard on contact: at least 20 balls in play and xwOBAcon .040+ above league.** Separates pitches that get hurt on contact from pitches that lose value through walks or bad counts.

For scale: a pitch thrown 500 times at +1.0 runs per 100 is worth about 5 runs over a season.

In [ ]:
FLAG_RULES = [
    ('Underused weapon',             lambda r: r['whiff_plus'] >= 120 and r['rv_vs_lg'] >= 1.0 and r['usage_pct'] < 25),
    ('Misses bats, still gets hurt', lambda r: r['whiff_plus'] >= 120 and r['rv_vs_lg'] <= -1.0),
    ('Below league average',         lambda r: r['rv_vs_lg'] <= -1.5),
    ('Heavy use, below average',     lambda r: r['usage_pct'] >= 25 and r['rv_vs_lg'] < 0),
    ('Hit hard on contact',          lambda r: r['bip'] >= 20 and r['xwobacon_vs_lg'] >= 0.040),
]

def flag_pitcher(report):
    rows = []
    for (pitch, side), r in report.iterrows():
        label = f"{pitch} vs {'LHH' if side == 'L' else 'RHH'}"

        if r['small_sample']:
            rows.append((label, 'Small sample', f"only {int(r['swings'])} swings, too few to grade"))
            continue

        hits = [name for name, rule in FLAG_RULES if rule(r)]
        if hits:
            stats = (f"whiff+ {r['whiff_plus']:.0f}, {r['rv_vs_lg']:+.2f} runs/100 vs league, "
                     f"used {r['usage_pct']:.0f}%, xwOBAcon {r['xwobacon']:.3f} vs {r['xwobacon_lg']:.3f}")
            rows.append((label, ' + '.join(hits), stats))

    return pd.DataFrame(rows, columns=['pitch', 'flag', 'why'])

## What makes him nasty?
For each pitcher, the pitch that fools hitters most (highest combined whiff and chase percentile among pitches thrown 100+ times and 8%+ of the time). The answer names only traits ranking in the top 25% for that pitch type and throwing hand.

In [ ]:
import numpy as np

PITCH_NAMES = {'FF': 'four-seamer', 'SI': 'sinker', 'FC': 'cutter', 'SL': 'slider', 'ST': 'sweeper',
               'SV': 'slurve', 'CU': 'curveball', 'KC': 'knuckle curve', 'CS': 'slow curve', 'CH': 'changeup',
               'FS': 'splitter', 'FO': 'forkball', 'SC': 'screwball', 'KN': 'knuckleball', 'EP': 'eephus', 'FA': 'fastball'}

SHAPE_GOOD = {
    'FF': {'ivb': 1, 'hb': 0}, 'FC': {'ivb': 0, 'hb': -1}, 'SI': {'ivb': -1, 'hb': 1},
    'CH': {'ivb': -1, 'hb': 1}, 'FS': {'ivb': -1, 'hb': 0}, 'FO': {'ivb': -1, 'hb': 0},
    'SL': {'ivb': 0, 'hb': -1}, 'ST': {'ivb': 0, 'hb': -1}, 'SV': {'ivb': -1, 'hb': -1},
    'CU': {'ivb': -1, 'hb': 0}, 'KC': {'ivb': -1, 'hb': 0}, 'CS': {'ivb': -1, 'hb': 0},
}

def ordinal(n):
    n = int(round(n))
    suffix = 'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return f"{n}{suffix}"

def build_arsenal(reports):
    r = reports.copy()
    r['whiffs'] = r['whiff_pct'] * r['swings'] / 100
    r['w_velo'] = r['velo'] * r['pitches']
    r['w_ivb'] = r['ivb'] * r['pitches']
    r['w_hb'] = r['hb_arm'] * r['pitches']
    r['w_chase'] = r['chase_pct'] * r['pitches']
    g = r.groupby(['pitcher_id', 'p_throws', 'pitch_type'])
    a = pd.DataFrame({
        'pitches': g['pitches'].sum(), 'swings': g['swings'].sum(), 'whiffs': g['whiffs'].sum(),
        'velo': g['w_velo'].sum() / g['pitches'].sum(), 'ivb': g['w_ivb'].sum() / g['pitches'].sum(),
        'hb': g['w_hb'].sum() / g['pitches'].sum(), 'chase': g['w_chase'].sum() / g['pitches'].sum(),
    }).reset_index()
    a['whiff'] = a['whiffs'] / a['swings'] * 100
    a['usage'] = a['pitches'] / a.groupby('pitcher_id')['pitches'].transform('sum') * 100
    pool = a['pitches'] >= 100
    for col in ['velo', 'whiff', 'chase']:
        a[f'{col}_pct'] = a[pool].groupby(['pitch_type', 'p_throws'])[col].rank(pct=True) * 100
    for col in ['ivb', 'hb']:
        sign = a['pitch_type'].map(lambda t: SHAPE_GOOD.get(t, {}).get(col, 0))
        a[f'{col}_good'] = a[col] * sign
        a[f'{col}_pct'] = a[pool & (sign != 0)].groupby(['pitch_type', 'p_throws'])[f'{col}_good'].rank(pct=True) * 100
    return a

def trait_phrases(p, hand):
    who = f"{'lefty' if hand == 'L' else 'righty'} {PITCH_NAMES.get(p['pitch_type'], p['pitch_type'])}s"
    ivb_dir = SHAPE_GOOD.get(p['pitch_type'], {}).get('ivb', 0)
    traits = []
    if p['velo_pct'] >= 75:
        traits.append((p['velo_pct'], f"{p['velo']:.1f} mph ({ordinal(p['velo_pct'])} percentile among {who})"))
    if p['ivb_pct'] >= 75:
        if ivb_dir > 0:
            text = f"{p['ivb']:.1f}\" of carry"
        elif p['pitch_type'] in ('SI', 'CH', 'FS', 'FO'):
            text = f"heavy sink: {p['ivb']:.1f}\" of induced vertical break"
        else:
            text = f"{abs(p['ivb']):.1f}\" of drop"
        traits.append((p['ivb_pct'], f"{text} ({ordinal(p['ivb_pct'])} percentile)"))
    if p['hb_pct'] >= 75:
        word = 'of arm-side run' if p['hb'] > 0 else 'of sweep'
        traits.append((p['hb_pct'], f"{abs(p['hb']):.1f}\" {word} ({ordinal(p['hb_pct'])} percentile)"))
    return [t for _, t in sorted(traits, reverse=True)][:2]

def result_phrase(p):
    whiff = f"hitters miss on {p['whiff']:.0f}% of their swings"
    if p['whiff_pct'] >= 60:
        whiff += f" ({ordinal(p['whiff_pct'])} percentile)"
    chase = f"chase it {p['chase']:.0f}% of the time out of the zone"
    if p['chase_pct'] >= 60:
        chase += f" ({ordinal(p['chase_pct'])} percentile)"
    return f"{whiff} and {chase}"

def nasty(arsenal, pid):
    mine = arsenal[(arsenal['pitcher_id'] == pid) & (arsenal['pitches'] >= 100) & (arsenal['usage'] >= 8)]
    if mine.empty:
        return ""
    hand = mine['p_throws'].iloc[0]
    mine = mine.assign(score=mine[['whiff_pct', 'chase_pct']].fillna(0).mean(axis=1))
    top = mine.sort_values('score', ascending=False)
    p = top.iloc[0]
    name = PITCH_NAMES.get(p['pitch_type'], p['pitch_type'])
    traits = trait_phrases(p, hand)
    if traits:
        s = f"His {name}. It comes in with {' and '.join(traits)}, and {result_phrase(p)}."
    else:
        s = f"His {name}. Its speed and movement are close to average, but it plays: {result_phrase(p)}."
    if len(top) > 1:
        q = top.iloc[1]
        if q['score'] >= 75:
            qn = PITCH_NAMES.get(q['pitch_type'], q['pitch_type'])
            qt = trait_phrases(q, hand)
            extra = f" with {qt[0]}" if qt else ""
            s += f" The {qn} backs it up{extra}, missing bats on {q['whiff']:.0f}% of swings."
    return s

In [ ]:
def pitch_mix_report(first, last, season=2026):
    if season != 2026:
        print("The league baseline is built from 2026 only. Pick a 2026 season for now.")
        return None, None

    df = get_pitcher_data(first, last, season)
    if df is None or df.empty:
        return None, None

    report = grade_pitcher(df).sort_values(['stand', 'usage_pct'], ascending=[True, False])
    flags = flag_pitcher(report)

    display(report[['usage_pct', 'swings', 'whiff_plus', 'rv_vs_lg', 'bip', 'hard_hit_pct', 'xwobacon_vs_lg',
                    'zone_pct', 'chase_pct', 'chase_vs_lg', 'z_whiff_vs_lg', 'small_sample']])
    display(flags)
    return report, flags

In [ ]:
miz_report, miz_flags = pitch_mix_report('jacob', 'misiorowski')

Gathering Player Data
Jacob Misiorowski (2026): 2649 regular-season pitches


,,usage_pct,swings,whiff_plus,rv_vs_lg,bip,hard_hit_pct,xwobacon_vs_lg,zone_pct,chase_pct,chase_vs_lg,z_whiff_vs_lg,small_sample
pitch_type,stand,,,,,,,,,,,,
FF,L,63.9,528,245.0,2.41,90,36.67,-0.061,54.23,36.99,10.3,23.7,False
FC,L,14.8,113,119.0,4.03,36,30.56,-0.164,48.20,38.26,9.3,-6.2,False
CU,L,12.9,92,140.0,3.63,25,28.00,-0.095,39.90,34.48,5.0,1.3,False
SL,L,6.5,56,119.0,0.15,16,31.25,-0.14,44.90,40.74,6.3,-0.2,False
CH,L,1.7,10,72.0,5.56,5,20.00,-0.064,34.62,23.53,-10.5,-18.2,True
FF,R,62.6,331,129.0,0.95,90,42.22,-0.01,51.94,30.92,1.9,0.8,False
SL,R,17.3,119,100.0,2.43,49,26.53,-0.041,49.75,43.00,9.6,-1.8,False
FC,R,12.8,69,77.0,0.75,34,32.35,-0.009,53.74,20.59,-7.9,-5.0,False
CU,R,7.0,39,148.0,1.27,15,6.67,-0.113,28.75,42.11,10.3,-6.8,False


,pitch,flag,why
0,CU vs LHH,Underused weapon,"whiff+ 140, +3.63 runs/100 vs league, used 13%, xwOBAcon 0.273 vs 0.368"
1,CH vs LHH,Small sample,"only 10 swings, too few to grade"
2,CU vs RHH,Underused weapon,"whiff+ 148, +1.27 runs/100 vs league, used 7%, xwOBAcon 0.227 vs 0.340"


In [ ]:
sanchez_report, sanchez_flags = pitch_mix_report('cristopher', 'sanchez')

Gathering Player Data
Cristopher Sanchez (2026): 3158 regular-season pitches


usage_pct  swings  whiff_plus  rv_vs_lg  bip  hard_hit_pct  \
pitch_type stand                                                               
SI         L           58.6     210       126.0      3.04   90         38.89   
SL         L           28.5      98       149.0      2.54   25         12.00   
CH         L           12.9      57       170.0      4.49    6         50.00   
           R           43.4     584       138.0      1.02  174         27.59   
SI         R           40.7     453        61.0     -1.53  220         59.55   
SL         R           15.9     192       120.0     -0.14   59         35.59   

                  xwobacon_vs_lg  zone_pct  chase_pct  chase_vs_lg  \
pitch_type stand                                                     
SI         L              -0.032     62.60      39.58          9.9   
SL         L              -0.068     38.50      42.61         10.3   
CH         L               0.009     31.76      56.90         19.3   
           R              -0.042     32.69      42.27          6.1   
SI         R               0.074     59.10      18.75         -2.2   
SL         R               0.033     47.49      39.23          4.0   

                  z_whiff_vs_lg  small_sample  
pitch_type stand                               
SI         L                1.3         False  
SL         L               10.4         False  
CH         L               28.8         False  
           R                8.4         False  
SI         R               -3.0         False  
SL         R                3.1         False

,pitch,flag,why
0,CH vs LHH,Underused weapon,"whiff+ 170, +4.49 runs/100 vs league, used 13%, xwOBAcon 0.396 vs 0.387"
1,SI vs RHH,"Below league average + Heavy use, below average + Hit hard on contact","whiff+ 61, -1.53 runs/100 vs league, used 41%, xwOBAcon 0.452 vs 0.378"


In [ ]:
miller_report, miller_flags = pitch_mix_report('mason', 'miller')

Gathering Player Data
Mason Miller (2026): 1154 regular-season pitches


,,usage_pct,swings,whiff_plus,rv_vs_lg,bip,hard_hit_pct,xwobacon_vs_lg,zone_pct,chase_pct,chase_vs_lg,z_whiff_vs_lg,small_sample
pitch_type,stand,,,,,,,,,,,,
SL,L,49.3,122,191.0,2.59,19,21.05,-0.166,43.71,36.31,1.9,18.4,False
FF,L,47.3,164,155.0,1.69,42,40.48,0.001,48.20,32.91,6.2,6.2,False
CH,L,3.4,7,51.0,-11.78,2,50.00,0.454,13.64,21.05,-13.0,-18.2,True
SL,R,50.2,125,161.0,1.72,27,7.41,-0.113,48.22,48.09,14.7,-0.2,False
FF,R,49.6,124,204.0,2.69,25,36.00,-0.1,42.80,30.07,1.1,21.4,False


,pitch,flag,why
0,CH vs LHH,Small sample,"only 7 swings, too few to grade"


## Findings (2026)
**Tarik Skubal.** His four-seamer misses bats against lefties at almost twice the league rate (whiff+ 182) but still costs runs (−1.42 per 100). Lefties have put only 13 of them in play, but those were hit hard (xwOBAcon .547 vs. .384 league), which points to damage on contact rather than only walks or counts. By count, his changeup is his go-to when ahead (31.5% usage, whiff+ 166) but saves the most runs when he's behind (+3.09). His slider flips the other way: +3.04 behind, −1.97 ahead. Is the slider ahead in the count a location problem or a predictability problem?

**Jacob Misiorowski.** The four-seamer carries him: 64% usage against lefties, 100.6 mph, whiff+ 245. The curveball is flagged as underused against both sides (13% vs. LHH, 7% vs. RHH). Does it keep that value if he throws it more, or does it depend on hitters sitting on the fastball?

**Cristopher Sánchez.** Against righties, the changeup is his main pitch (43%) and grades above average. The sinker is used almost as much (41%) but grades poorly (whiff+ 61, −1.53), and the contact data says why: righties hit it 95+ mph 60% of the time, against 48% for league lefties' sinkers to righties. His sinker and changeup share almost the same arm-side run (about 17"), separated mostly by speed (95 vs. 87 mph). Does that shared shape help the changeup and hurt the sinker?

**Mason Miller.** No flags beyond a small sample. Slider and four-seamer split almost evenly against both sides, and all four grade well above league average. For an elite two-pitch reliever, the useful answer is confirmation: nothing to change.

In [ ]:
all_reports = []
all_flags = []

for pid, df in league.groupby('pitcher'):
    if len(df) < 300:
        continue
    name = df['player_name'].iloc[0]
    hand = df['p_throws'].mode().iloc[0]
    rep = grade_pitcher(df)
    if rep.empty:
        continue
    fl = flag_pitcher(rep)

    rep = rep.reset_index()
    rep.insert(0, 'pitcher', name)
    rep.insert(1, 'pitcher_id', pid)
    rep.insert(2, 'p_throws', hand)
    fl.insert(0, 'pitcher', name)
    fl.insert(1, 'pitcher_id', pid)

    all_reports.append(rep)
    all_flags.append(fl)

reports = pd.concat(all_reports, ignore_index=True)
flags = pd.concat(all_flags, ignore_index=True)
print(f"{reports['pitcher'].nunique()} pitchers graded")

563 pitchers graded


In [ ]:
reports.to_csv(f'{SAVE_DIR}/all_report_cards_2026.csv', index=False)
flags.to_csv(f'{SAVE_DIR}/all_flags_2026.csv', index=False)

from google.colab import files
files.download(f'{SAVE_DIR}/all_report_cards_2026.csv')
files.download(f'{SAVE_DIR}/all_flags_2026.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

all_counts = []
for pid, df in league.groupby('pitcher'):
    if len(df) < 300:
        continue
    c = count_report(df, wide=False)
    if c.empty:
        continue
    c.insert(0, 'pitcher', df['player_name'].iloc[0])
    c.insert(1, 'pitcher_id', pid)
    all_counts.append(c)

counts = pd.concat(all_counts, ignore_index=True)
counts.to_csv(f'{SAVE_DIR}/all_count_reports_2026.csv', index=False)
print(f"{counts['pitcher'].nunique()} pitchers, {len(counts)} rows")
files.download(f'{SAVE_DIR}/all_count_reports_2026.csv')

563 pitchers, 7253 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Limits
- **2026 only.** The baseline is built from one season, so the tool only grades 2026 pitchers for now.
- **Small samples are flagged, not graded.** Piles under 30 swings, contact numbers under 20 balls in play, and pitches thrown fewer than 10 times to a side are held out.
- **Run value is noisy.** On a few hundred pitches, run value moves a lot. Flags are prompts to look closer, not verdicts.
- **Rare pitches can look good because they're rare.** A pitch thrown 7% of the time is often thrown in its best spots. The same applies to counts: a pitch used more when ahead faces hitters who expect it. "Underused weapon" means worth testing, not proven.
- **Whiff+ inflates on low-whiff pitches.** Sinkers have a low league whiff rate, so a few extra misses produce a very high whiff+.
- **Count context is split from handedness.** Splitting by both would leave most piles too small, so the count view combines both batter sides.
- **It can't prove a new mix would work.** The tool shows where usage and results don't line up. Hitters adjust when a pitcher changes his mix.
- **Pitch labels can change between seasons.** Statcast sometimes reclassifies a pitch (a changeup becoming a splitter, a slider becoming a sweeper), which shows up as a "Dropped pitch" and a "New pitch" on the same pitcher. Check pitch shape before calling it a real change.

## Year over year (2025 → 2026)
Each pitcher's arsenal is compared to his own 2025 season (both seasons 300+ pitches). A pitch is flagged for:
- **New / Dropped:** under 10 thrown one season, 5%+ usage the other
- **Usage up / down:** a change of 10+ percentage points
- **Velo up / down:** 1.5+ mph
- **Shape change:** 2.5+ inches of vertical or horizontal break

Changes are only measured when a pitch has 20+ thrown in both seasons.

In [ ]:
months_2025 = [('2025-03-15', '2025-04-30'), ('2025-05-01', '2025-05-31'),
               ('2025-06-01', '2025-06-30'), ('2025-07-01', '2025-07-31'),
               ('2025-08-01', '2025-08-31'), ('2025-09-01', '2025-09-30')]

for start, end in months_2025:
    path = f'{SAVE_DIR}/league_2025_{start[:7]}.parquet'
    if os.path.exists(path):
        print(f'{start[:7]} already saved, skipping')
        continue
    chunk = statcast(start, end)
    chunk = chunk[chunk['game_type'] == 'R']
    chunk = chunk[[c for c in KEEP if c in chunk.columns]]
    chunk.to_parquet(path)
    print(f'{start[:7]}: {len(chunk):,} pitches saved')

league_2025 = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(f'{SAVE_DIR}/league_2025_*.parquet'))],
                        ignore_index=True)
league_2025 = add_flags(league_2025)
print(f'{len(league_2025):,} pitches in 2025')

2025-03 already saved, skipping
2025-05 already saved, skipping
2025-06 already saved, skipping
2025-07 already saved, skipping
2025-08 already saved, skipping
2025-09 already saved, skipping
712,528 pitches in 2025


In [ ]:
def arsenal(df):
    g = df.groupby(['pitcher', 'pitch_type'])
    out = pd.DataFrame({
        'pitches': g.size(),
        'velo': g['release_speed'].mean(),
        'ivb': g['ivb'].mean(),
        'hb_arm': g['hb_arm'].mean(),
        'swings': g['is_swing'].sum(),
        'whiffs': g['is_whiff'].sum(),
        'rv_per_100': g['rv_pitcher'].mean() * 100,
    })
    out['usage_pct'] = out['pitches'] / out.groupby(level='pitcher')['pitches'].transform('sum') * 100
    out['whiff_pct'] = out['whiffs'] / out['swings'] * 100
    return out.astype(float)

a25 = arsenal(league_2025).add_suffix('_25')
a26 = arsenal(league).add_suffix('_26')
yoy = a26.join(a25, how='outer')

tot25 = league_2025.groupby('pitcher').size()
tot26 = league.groupby('pitcher').size()
both = tot25[tot25 >= 300].index.intersection(tot26[tot26 >= 300].index)
yoy = yoy[yoy.index.get_level_values('pitcher').isin(both)].copy()

for col in ['pitches_25', 'pitches_26', 'usage_pct_25', 'usage_pct_26']:
    yoy[col] = yoy[col].fillna(0)

yoy['usage_chg'] = yoy['usage_pct_26'] - yoy['usage_pct_25']
enough = (yoy['pitches_25'] >= 20) & (yoy['pitches_26'] >= 20)
for col in ['velo', 'ivb', 'hb_arm', 'whiff_pct', 'rv_per_100']:
    yoy[f'{col}_chg'] = (yoy[f'{col}_26'] - yoy[f'{col}_25']).where(enough)

CHANGE_RULES = [
    ('New pitch',     lambda r: r['pitches_25'] < 10 and r['usage_pct_26'] >= 5),
    ('Dropped pitch', lambda r: r['pitches_26'] < 10 and r['usage_pct_25'] >= 5),
    ('Usage up',      lambda r: r['usage_chg'] >= 10),
    ('Usage down',    lambda r: r['usage_chg'] <= -10),
    ('Velo up',       lambda r: r['velo_chg'] >= 1.5),
    ('Velo down',     lambda r: r['velo_chg'] <= -1.5),
    ('Shape change',  lambda r: abs(r['ivb_chg']) >= 2.5 or abs(r['hb_arm_chg']) >= 2.5),
]
yoy['changes'] = yoy.apply(lambda r: ' + '.join(name for name, rule in CHANGE_RULES if rule(r)), axis=1)

names = league.groupby('pitcher')['player_name'].first()
yoy = yoy.reset_index()
yoy.insert(1, 'player_name', yoy['pitcher'].map(names))
yoy = yoy.round(2)

print(f"{yoy['pitcher'].nunique()} pitchers with 300+ pitches in both seasons")
cols = ['pitch_type', 'usage_pct_25', 'usage_pct_26', 'velo_chg', 'ivb_chg', 'hb_arm_chg', 'whiff_pct_chg', 'rv_per_100_chg', 'changes']
yoy[yoy['player_name'] == 'Skubal, Tarik'][cols]

398 pitchers with 300+ pitches in both seasons


,pitch_type,usage_pct_25,usage_pct_26,velo_chg,ivb_chg,hb_arm_chg,whiff_pct_chg,rv_per_100_chg,changes
1202,CH,31.41,25.59,-0.39,-1.72,-1.29,0.66,-1.81,
1203,CU,2.77,5.01,-0.78,0.28,-3.76,6.63,0.12,Shape change
1204,FF,29.31,38.03,-0.77,0.06,-0.29,-2.43,-0.14,
1205,FS,0.11,0.00,NaN,NaN,NaN,NaN,NaN,
1206,SI,23.90,17.49,-0.59,-1.27,0.08,-0.65,-0.42,
1207,SL,12.50,13.88,-0.25,-0.74,-0.02,1.74,-0.08,


In [ ]:
from google.colab import files

yoy.to_csv(f'{SAVE_DIR}/yoy_changes_2025_2026.csv', index=False)
print(yoy['changes'].str.len().gt(0).sum(), 'pitches flagged with a change')
files.download(f'{SAVE_DIR}/yoy_changes_2025_2026.csv')

775 pitches flagged with a change


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests, time

API = 'https://statsapi.mlb.com/api/v1'

teams = requests.get(f'{API}/teams', params={'sportIds': '1,11,12,13,14,16'}, timeout=30).json()['teams']
team_name = {t['id']: t['name'] for t in teams}
team_org = {t['id']: t.get('parentOrgId', t['id']) for t in teams}

ids = [int(i) for i in reports['pitcher_id'].unique()]
people = []
for i in range(0, len(ids), 100):
    batch = ids[i:i + 100]
    resp = requests.get(f'{API}/people',
                        params={'personIds': ','.join(map(str, batch)), 'hydrate': 'currentTeam,education'},
                        timeout=30)
    resp.raise_for_status()
    people.extend(resp.json().get('people', []))
    time.sleep(0.5)
print(f'{len(people)} players found')

ACQ_WORDS = ('Trade', 'Free Agent', 'Waiver', 'Purchase', 'Rule 5', 'Draft')

def how_acquired(pid, org_id):
    if org_id is None:
        return ''
    resp = requests.get(f'{API}/transactions', params={'playerId': pid}, timeout=30)
    if resp.status_code != 200:
        return ''
    moves = []
    for t in resp.json().get('transactions', []):
        to_id = t.get('toTeam', {}).get('id')
        if to_id is None or team_org.get(to_id, to_id) != org_id:
            continue
        if any(word in t.get('typeDesc', '') for word in ACQ_WORDS):
            moves.append(t)
    if not moves:
        return ''
    last = sorted(moves, key=lambda t: t.get('date', ''))[-1]
    return f"{last.get('description', '').strip()} ({last.get('date', '')[:4]})"

bios = []
for n, p in enumerate(people):
    team_id = p.get('currentTeam', {}).get('id')
    org_id = team_org.get(team_id, team_id) if team_id else None
    colleges = [c.get('name', '') for c in p.get('education', {}).get('colleges', [])]
    bios.append({
        'pitcher_id': p['id'],
        'name': p.get('fullName', ''),
        'height': p.get('height', ''),
        'weight': p.get('weight', ''),
        'college': ', '.join(colleges),
        'team': team_name.get(org_id, ''),
        'how_acquired': how_acquired(p['id'], org_id),
    })
    time.sleep(0.2)
    if n % 50 == 0:
        print(f'{n} of {len(people)} done')

bios = pd.DataFrame(bios)
bios.to_csv(f'{SAVE_DIR}/bios_2026.csv', index=False)
print(bios.head(10))

from google.colab import files
files.download(f'{SAVE_DIR}/bios_2026.csv')

563 players found
0 of 563 done
50 of 563 done
100 of 563 done
150 of 563 done
200 of 563 done
250 of 563 done
300 of 563 done
350 of 563 done
400 of 563 done
450 of 563 done
500 of 563 done
550 of 563 done
   pitcher_id           name  height  weight              college  \
0      445276  Kenley Jansen   6' 5"     265                        
1      453286   Max Scherzer   6' 3"     208             Missouri   
2      476594   Robert Stock   6' 1"     260  Southern California   
3      489446    Kirby Yates  5' 10"     205           Yavapai JC   
4      500779  Jose Quintana   6' 2"     224                        
5      518585  Fernando Cruz   6' 2"     237                        
6      518876  Merrill Kelly   6' 2"     210        Arizona State   
7      518886  Craig Kimbrel   6' 0"     215     Wallace State CC   
8      519141  Drew Pomeranz   6' 5"     246          Mississippi   
9      519242     Chris Sale   6' 6"     180   Florida Gulf Coast   

                   team  \
0     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests, time

API = 'https://statsapi.mlb.com/api/v1'
teams = requests.get(f'{API}/teams', params={'sportIds': '1,11,12,13,14,16'}, timeout=30).json()['teams']
team_org = {t['id']: t.get('parentOrgId', t['id']) for t in teams}
org_by_name = {t['name']: t['id'] for t in teams if t.get('sport', {}).get('id') == 1}

bios['how_acquired'] = bios['how_acquired'].fillna('')
blank = bios[bios['how_acquired'] == '']
print(f'{len(blank)} pitchers to fill')

def first_signing(pid, org_id):
    resp = requests.get(f'{API}/transactions', params={'playerId': pid}, timeout=30)
    if resp.status_code != 200:
        return ''
    moves = []
    for t in resp.json().get('transactions', []):
        to_id = t.get('toTeam', {}).get('id')
        if to_id is None or team_org.get(to_id, to_id) != org_id:
            continue
        if any(word in t.get('typeDesc', '') for word in ('Signed', 'Draft', 'Free Agent')):
            moves.append(t)
    if not moves:
        return ''
    first = sorted(moves, key=lambda t: t.get('date', ''))[0]
    return f"{first.get('description', '').strip()} ({first.get('date', '')[:4]})"

filled = 0
for idx, row in blank.iterrows():
    org_id = org_by_name.get(row['team'])
    person = requests.get(f"{API}/people/{row['pitcher_id']}", timeout=30).json().get('people', [{}])[0]
    draft_year = person.get('draftYear')
    text = first_signing(row['pitcher_id'], org_id) if org_id else ''
    if draft_year and text:
        text = f"Drafted in {draft_year}. {text}"
    elif draft_year:
        text = f"Drafted in {draft_year}."
    if text:
        bios.at[idx, 'how_acquired'] = text
        filled += 1
    time.sleep(0.2)

print(f'filled {filled} of {len(blank)}')
print(bios.loc[blank.index, ['name', 'how_acquired']].head(15))

bios.to_csv(f'{SAVE_DIR}/bios_2026.csv', index=False)
from google.colab import files
files.download(f'{SAVE_DIR}/bios_2026.csv')

129 pitchers to fill
filled 128 of 129
                 name  \
68      Kyle Freeland   
116       Bailey Ober   
147      Kevin Ginkel   
152      Mitch Keller   
154        Jacob Latz   
165        Logan Webb   
183      Ryan Feltner   
185      Davis Martin   
189  Shane McClanahan   
194        Jake Irvin   
195      Hogan Harris   
196   Daniel Lynch IV   
226       Cole Ragans   
227       Nick Lodolo   
238         Cole Winn   

                                                              how_acquired  
68      Drafted in 2014. Colorado Rockies signed LHP Kyle Freeland. (2014)  
116        Drafted in 2017. Minnesota Twins signed RHP Bailey Ober. (2017)  
147  Drafted in 2016. Arizona Diamondbacks signed RHP Kevin Ginkel. (2016)  
152    Drafted in 2014. Pittsburgh Pirates signed RHP Mitch Keller. (2014)  
154            Drafted in 2017. Texas Rangers signed LHP Jake Latz. (2017)  
165    Drafted in 2014. San Francisco Giants signed RHP Logan Webb. (2014)  
183      Drafted in 2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Arm angle
Average arm angle at release from Statcast, per pitcher per season (300+ pitches). A change of **4+ degrees** from 2025 to 2026 is flagged as a slot change. Slot names (sidearm, three-quarters and so on) are a rough guide, not an official scale.

In [ ]:
test = get_pitcher_data('tarik', 'skubal', 2026)
print('arm_angle' in test.columns)
print(test.groupby('pitch_type')['arm_angle'].mean().round(1))

Gathering Player Data
Tarik Skubal (2026): 2357 regular-season pitches
True
pitch_type
CH    42.0
CU    51.9
FF    51.4
SI    47.8
SL    45.9
Name: arm_angle, dtype: float64


In [ ]:
ARM_KEEP = ['pitcher', 'player_name', 'game_date', 'game_type', 'p_throws', 'pitch_type',
            'arm_angle', 'release_pos_x', 'release_pos_z', 'release_extension', 'release_speed', 'pfx_x', 'pfx_z']

seasons = {
    2025: [('2025-03-15', '2025-04-30'), ('2025-05-01', '2025-05-31'), ('2025-06-01', '2025-06-30'),
           ('2025-07-01', '2025-07-31'), ('2025-08-01', '2025-08-31'), ('2025-09-01', '2025-09-30')],
    2026: [('2026-03-15', '2026-04-30'), ('2026-05-01', '2026-05-31'), ('2026-06-01', '2026-06-30'),
           ('2026-07-01', '2026-07-31'), ('2026-08-01', '2026-08-31'), ('2026-09-01', '2026-09-30')],
}

for year, months_y in seasons.items():
    for start, end in months_y:
        path = f'{SAVE_DIR}/arm_{start[:7]}.parquet'
        if os.path.exists(path):
            print(f'{start[:7]} already saved, skipping')
            continue
        chunk = statcast(start, end)
        chunk = chunk[chunk['game_type'] == 'R']
        chunk = chunk[[c for c in ARM_KEEP if c in chunk.columns]]
        chunk['season'] = year
        chunk.to_parquet(path)
        print(f'{start[:7]}: {len(chunk):,} pitches saved')

arm = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(f'{SAVE_DIR}/arm_*.parquet'))], ignore_index=True)
print(f'{len(arm):,} pitches with release data')

2025-03 already saved, skipping
2025-05 already saved, skipping
2025-06 already saved, skipping
2025-07 already saved, skipping
2025-08 already saved, skipping
2025-09 already saved, skipping
2026-03 already saved, skipping
2026-05 already saved, skipping
2026-06 already saved, skipping
2026-07 already saved, skipping
2026-08 already saved, skipping
2026-09 already saved, skipping
1,429,320 pitches with release data


In [ ]:
arm['arm_angle'] = arm['arm_angle'].astype(float)

def slot_label(a):
    if a >= 60: return 'Over the top'
    if a >= 45: return 'High three-quarters'
    if a >= 30: return 'Three-quarters'
    if a >= 15: return 'Low three-quarters'
    if a >= 0:  return 'Sidearm'
    return 'Submarine'

by_year = (arm.dropna(subset=['arm_angle'])
              .groupby(['pitcher', 'season'])
              .agg(arm_angle=('arm_angle', 'mean'), pitches=('arm_angle', 'size'),
                   rel_height=('release_pos_z', 'mean'), extension=('release_extension', 'mean'))
              .reset_index())
by_year = by_year[by_year['pitches'] >= 300]

wide = by_year.pivot(index='pitcher', columns='season', values=['arm_angle', 'rel_height', 'extension'])
wide.columns = [f'{col}_{yr}' for col, yr in wide.columns]
wide = wide.reset_index()

wide['arm_change'] = wide['arm_angle_2026'] - wide['arm_angle_2025']
wide['slot_2026'] = wide['arm_angle_2026'].map(lambda a: slot_label(a) if pd.notna(a) else '')
wide['slot_change_flag'] = wide['arm_change'].abs() >= 4

names = arm.groupby('pitcher')['player_name'].first()
wide.insert(1, 'player_name', wide['pitcher'].map(names))
wide = wide.round(2)

print(f"{wide['arm_angle_2026'].notna().sum()} pitchers with a 2026 arm angle")
print(f"{wide['slot_change_flag'].sum()} changed their slot by 4+ degrees")
wide[wide['slot_change_flag']].sort_values('arm_change')[['player_name', 'arm_angle_2025', 'arm_angle_2026', 'arm_change', 'slot_2026']].head(15)

561 pitchers with a 2026 arm angle
61 changed their slot by 4+ degrees


,player_name,arm_angle_2025,arm_angle_2026,arm_change,slot_2026
651,"Klein, Will",53.577705,42.795276,-10.782429,Three-quarters
397,"Allen, Logan",36.198892,26.124615,-10.074276,Low three-quarters
715,"Tong, Jonah",64.353639,54.943414,-9.410225,High three-quarters
469,"Weathers, Ryan",41.524766,32.773628,-8.751138,Three-quarters
513,"Burrows, Mike",47.660833,39.596863,-8.06397,Three-quarters
456,"Cavalli, Cade",43.897081,36.043383,-7.853699,Three-quarters
432,"Hancock, Emerson",18.515952,11.291939,-7.224013,Sidearm
583,"Young, Brandon",48.932375,41.999684,-6.932691,Three-quarters
430,"Junk, Janson",52.567448,45.817769,-6.749679,High three-quarters
197,"Soroka, Michael",32.818194,26.473489,-6.344704,Low three-quarters


In [ ]:
from google.colab import files
wide.to_csv(f'{SAVE_DIR}/arm_angles_2025_2026.csv', index=False)
files.download(f'{SAVE_DIR}/arm_angles_2025_2026.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
num_cols = reports.select_dtypes('number').columns
clean = reports.copy()
clean[num_cols] = clean[num_cols].astype(float)

arsenal = build_arsenal(clean)
pct_cols = [c for c in arsenal.columns if c.endswith('_pct')]
arsenal[pct_cols] = arsenal[pct_cols].astype(float).fillna(0)

nasty_df = pd.DataFrame({'pitcher_id': reports['pitcher_id'].unique()})
nasty_df['what_makes_him_nasty'] = nasty_df['pitcher_id'].map(lambda pid: nasty(arsenal, pid))
nasty_df.to_csv(f'{SAVE_DIR}/nasty_2026.csv', index=False)
print(nasty(arsenal, 669373))

His changeup. It comes in with 87.7 mph (88th percentile among lefty changeups), and hitters miss on 46% of their swings (96th percentile) and chase it 46% of the time out of the zone (95th percentile). The four-seamer backs it up with 96.8 mph (96th percentile among lefty four-seamers), missing bats on 23% of swings.


## Limits
- **One season of baseline.** League comparisons use 2026 only.
- **Small samples are flagged, not graded.** Under 30 swings, under 20 balls in play for contact, and under 10 pitches to a side are held out.
- **Run value is noisy.** Flags are prompts to look closer, not verdicts.
- **Rare pitches can look good because they're rare.** The same applies to counts.
- **Whiff+ inflates on low-whiff pitches** like sinkers.
- **Pitch labels can change between seasons.** A "Dropped" and a "New" pitch on the same pitcher can be a relabel. Check pitch shape and arm angle before calling it a real change.
- **Arm angle is estimated** from body tracking.
- **It can't prove a new mix would work.** Hitters adjust.